In [ ]:
import os
import cv2
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

tqdm.pandas()

# Dataset Preparation


## Image Preprocessing and Pickle Saving
Initially, each image in the dataset is loaded, resized to a standard shape (224x224), and converted into a numerical array (pixels) so that it can be used for training a deep learning model. This process is applied to every image in both the training and testing datasets, which can be time-consuming.

To avoid repeating this preprocessing step every time the model is trained or evaluated, the processed data is saved using pickle, a Python module for serializing objects. The train_df and test_df, now containing image pixels and file information, are saved as .pkl files to Google Drive. Later, these preprocessed files can be quickly loaded, saving time and computational resources during repeated experiments or model development.

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/Training/Dataset_080625/train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Training/Dataset_080625/val.csv")

In [ ]:
print("trainset consists of ",train_df.shape)
print("test set consist of ",test_df.shape)

trainset consists of  (1104, 2)
test set consist of  (276, 2)


In [ ]:
train_df = train_df[['file_name', 'ethnicity']]
test_df = test_df[['file_name', 'ethnicity']]

In [ ]:
train_df['file_name'] = '/content/drive/MyDrive/Training/Dataset_080625/img/train/'+train_df['file_name']
test_df['file_name'] = '/content/drive/MyDrive/Training/Dataset_080625/img/val/'+test_df['file_name']

In [ ]:
train_df.head()

,file_name,ethnicity
0,/content/drive/MyDrive/Training/Dataset_080625...,Indian
1,/content/drive/MyDrive/Training/Dataset_080625...,Malay
2,/content/drive/MyDrive/Training/Dataset_080625...,Chinese
3,/content/drive/MyDrive/Training/Dataset_080625...,Indian
4,/content/drive/MyDrive/Training/Dataset_080625...,Malay


In [ ]:
100*train_df.groupby(['ethnicity']).count()[['file_name']]/train_df.groupby(['ethnicity']).count()[['file_name']].sum()

,file_name
ethnicity,
Chinese,33.333333
Indian,33.333333
Malay,33.333333


In [ ]:
target_size = (224, 224)

def getImagePixels(file):
    #print(file)
    img = image.load_img(file, target_size=target_size)
    x = image.img_to_array(img).reshape(1, -1)[0]
    return x

In [ ]:
train_df['pixels'] = train_df['file_name'].progress_apply(getImagePixels)
test_df['pixels'] = test_df['file_name'].progress_apply(getImagePixels)

100%|██████████| 276/276 [01:34<00:00,  2.93it/s]


In [ ]:
# Save as pixels in pickle file

# Define the paths
train_save_path = '/content/drive/MyDrive/Training/Dataset_080625/train_df_preprocessed.pkl'
test_save_path = '/content/drive/MyDrive/Training/Dataset_080625/test_df_preprocessed.pkl'

# Save the DataFrames
with open(train_save_path, 'wb') as f:
    pickle.dump(train_df, f)

with open(test_save_path, 'wb') as f:
    pickle.dump(test_df, f)

print("Saved both train_df and test_df to Google Drive.")

In [ ]:
# skip getImagePixels (images preprocessing)
# Load .pkl file

# Load the saved DataFrames
with open('/content/drive/MyDrive/Training/Dataset_080625/train_df_preprocessed.pkl', 'rb') as f:
    train_df = pickle.load(f)

with open('/content/drive/MyDrive/Training/Dataset_080625/test_df_preprocessed.pkl', 'rb') as f:
    test_df = pickle.load(f)

In [ ]:
train_df.head()

,file_name,ethnicity,pixels
0,/content/drive/MyDrive/Training/Dataset_080625...,Indian,"[29.0, 24.0, 30.0, 30.0, 25.0, 31.0, 31.0, 26...."
1,/content/drive/MyDrive/Training/Dataset_080625...,Malay,"[199.0, 237.0, 246.0, 199.0, 237.0, 246.0, 199..."
2,/content/drive/MyDrive/Training/Dataset_080625...,Chinese,"[47.0, 29.0, 29.0, 48.0, 30.0, 30.0, 50.0, 32...."
3,/content/drive/MyDrive/Training/Dataset_080625...,Indian,"[8.0, 8.0, 6.0, 8.0, 8.0, 6.0, 8.0, 8.0, 6.0, ..."
4,/content/drive/MyDrive/Training/Dataset_080625...,Malay,"[78.0, 70.0, 68.0, 78.0, 70.0, 68.0, 77.0, 69...."
